# Lyman-alpha fion Animation Notebook

Create a movie of ionization evolution across redshift from `fion` slices.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
from tqdm.auto import tqdm
from IPython.display import Video, Image, display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from lyman_alpha.data import list_snapshots, load_ionized_fraction, parse_redshift

In [ ]:
data_dir = PROJECT_ROOT / 'data' / 'for_aryana' / 'late_end_early_start'
snapshots = list_snapshots(data_dir)
print('Snapshots:', len(snapshots))
print('First:', snapshots[0].name)
print('Last :', snapshots[-1].name)

In [ ]:
# Animation settings
z_stride = 2          # use every Nth snapshot to speed up
slice_axis = 2        # 0,1,2
slice_index = 100     # for N=200, center is 100
fps = 8
n_grid = 200

selected = snapshots[::z_stride]
print('Frames to render:', len(selected))

In [ ]:
def get_slice(cube, axis, idx):
    if axis == 0:
        return cube[idx, :, :]
    if axis == 1:
        return cube[:, idx, :]
    return cube[:, :, idx]

# Preview one frame
test = load_ionized_fraction(selected[0], n_grid=n_grid, memmap=False)
img = get_slice(test, slice_axis, slice_index)

plt.figure(figsize=(6, 5))
im = plt.imshow(img, origin='lower', cmap='magma', vmin=0, vmax=1)
plt.colorbar(im, label='fion')
plt.title(f'Preview: {selected[0].name}')
plt.tight_layout()
plt.show()

In [ ]:
# Render frames as RGB arrays
frames = []

for snap in tqdm(selected):
    z = parse_redshift(snap)
    fion = load_ionized_fraction(snap, n_grid=n_grid, memmap=False)
    plane = get_slice(fion, slice_axis, slice_index)

    fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
    im = ax.imshow(plane, origin='lower', cmap='magma', vmin=0, vmax=1)
    ax.set_title(f'fion slice, z={z:.4f}')
    ax.set_xlabel('cell index')
    ax.set_ylabel('cell index')
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.set_label('fion')
    fig.tight_layout()

    fig.canvas.draw()
    frame = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    frame = frame.reshape(fig.canvas.get_width_height()[::-1] + (4,))
    frames.append(frame[:, :, :3].copy())
    plt.close(fig)

print('Rendered frames:', len(frames))

In [ ]:
# Save movie
out_dir = PROJECT_ROOT / 'results'
out_dir.mkdir(parents=True, exist_ok=True)

mp4_path = out_dir / 'fion_evolution.mp4'
gif_path = out_dir / 'fion_evolution.gif'

imageio.mimwrite(mp4_path, frames, fps=fps, codec='libx264')
imageio.mimwrite(gif_path, frames, fps=fps)

print('saved:', mp4_path)
print('saved:', gif_path)

In [ ]:
# Inline preview
display(Video(str(mp4_path), embed=True, html_attributes='controls loop'))
display(Image(filename=str(gif_path)))